In [ ]:
import os
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

# Step 4.1: Establish high-resolution vector and figure export variables
scv.settings.set_figure_params(
    'scvelo', 
    dpi_save=300, 
    fontsize=10, 
    transparent=True
)

# Set global publication typography parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

data_path = "/mnt/d/._velocity_final.h5ad"
output_dir = "/mnt/d/.annotation plt"
os.makedirs(output_dir, exist_ok=True)

# Step 4.2: Data ingestion and dimension integrity check
print("Loading finalized velocity model matrix object...")
adata = sc.read_h5ad(data_path)
print(f"Dataset successfully synchronized. Current shape: {adata.shape}")
print(f"Active feature data layers: {list(adata.layers.keys())}")

# Step 4.3: Structural topology restoration graph computations
print("Re-calculating spatial neighborhood connectivity matrices...")
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)
sc.tl.umap(adata, min_dist=0.3) 

print("Running steady-state moment estimation and stochastic kinetics calculation...")
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)
scv.tl.velocity(adata, mode='stochastic')
scv.tl.velocity_graph(adata)

# Step 4.4: Comparative multi-panel stream visualization mapping
print("Generating side-by-side condition-stratified developmental velocity trajectories...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Slice object arrays by specific experimental cohorts
adata_mock = adata[adata.obs['Group'] == 'MOCK']
adata_zikv = adata[adata.obs['Group'] == 'ZIKV']

# Render reference baseline trajectory
scv.pl.velocity_embedding_stream(
    adata_mock, 
    basis='umap', 
    color='cell_type', 
    title='Control: MOCK Lineage Dynamics', 
    ax=ax1, 
    show=False, 
    legend_loc='none', 
    density=1.5, 
    arrow_size=1.5, 
    cmap='Spectral'
)

# Render pathological challenge topology trajectory using a matched layout scale
scv.pl.velocity_embedding_stream(
    adata_zikv, 
    basis='umap', 
    color='cell_type', 
    title='Pathological: ZIKV Lineage Trajectory Shifts', 
    ax=ax2, 
    show=False, 
    legend_loc='right margin', 
    density=1.5, 
    arrow_size=1.5, 
    cmap='Spectral'
)

plt.tight_layout()

# Save final graphic output to disk coordinates
save_plot_path = os.path.join(output_dir, "Differential_Velocity_Spectral_Colors.pdf")
plt.savefig(save_plot_path, format='pdf', bbox_inches='tight', dpi=300)
plt.close()

print(f"[SUCCESS] Differential velocity vectors successfully exported: {save_plot_path}")